In [1]:
import pandas as pd
import numpy as np
import sqlite3 
import os

In [2]:
sqliteConnection = sqlite3.connect('olist.db')

In [ ]:
data_dir = './data/'
l = list()

##cursor = sqliteConnection.cursor()
print('DB Init')
for file in os.listdir(data_dir): 
    #get a clean table name from the file
    table_name = file.replace("olist_","").replace("_dataset.csv","").replace(".csv","")
    
    path = os.path.join(data_dir,file)
    print(f'Loading {path} into {table_name}')
    temp_df = pd.read_csv(path)

    temp_df.to_sql(table_name,sqliteConnection,if_exists='replace',index=False)


DB Init
Loading ./data/olist_sellers_dataset.csv into sellers
Loading ./data/product_category_name_translation.csv into product_category_name_translation
Loading ./data/olist_orders_dataset.csv into orders
Loading ./data/olist_order_items_dataset.csv into order_items
Loading ./data/olist_customers_dataset.csv into customers
Loading ./data/olist_geolocation_dataset.csv into geolocation
Loading ./data/olist_order_payments_dataset.csv into order_payments
Loading ./data/olist_order_reviews_dataset.csv into order_reviews
Loading ./data/olist_products_dataset.csv into products


## Remembering basics of SQL

In [62]:
query="""
SELECT * FROM customers
LIMIT 10
"""
res = pd.read_sql_query(query,sqliteConnection)
res

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
6,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
7,5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
8,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
9,4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


In [63]:
query = """
SELECT COUNT(customer_id) AS total_customer
FROM customers
"""
res = pd.read_sql_query(query,sqliteConnection) #query that reads the total number of rows in the table
res

,total_customer
0,99441


In [64]:
query=""" 
SELECT AVG(payment_value) AS avg_payment
FROM order_payments
"""
pd.read_sql_query(query,sqliteConnection)

,avg_payment
0,154.10038


In [65]:
query = """ 
SELECT MAX(payment_value) as highest_payment,
        MIN(payment_value) as lower_payment
FROM order_payments
WHERE payment_value > 0.0
"""
pd.read_sql_query(query,sqliteConnection)

,highest_payment,lower_payment
0,13664.08,0.01


In [66]:
query = """ 
SELECT payment_type, ROUND(SUM(payment_value),2) as total_payment_type
FROM order_payments
WHERE payment_type != 'not_defined'
GROUP BY payment_type
ORDER BY total_payment_type DESC
"""
pd.read_sql_query(query,sqliteConnection)

,payment_type,total_payment_type
0,credit_card,12542084.19
1,boleto,2869361.27
2,voucher,379436.87
3,debit_card,217989.79


since we are aware that payment contain some rows with 0.0 values and payment type that is not defined (most likely coming from cancelled orders or some returns) we create a clean view of a table with those rows filtered out 

In [69]:
query = """ 
CREATE VIEW clean_table AS
SELECT * FROM order_payments
WHERE payment_type != 'not_defined'
"""
sqliteConnection.execute(query)
query_read = """
    SELECT * FROM clean_table 
    LIMIT 5;
"""
clean = pd.read_sql_query(query_read, sqliteConnection)
clean

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [71]:
query = """ 
SELECT payment_type, SUM(payment_value) AS total_value
FROM clean_table
GROUP BY payment_type
HAVING total_value > 1000000.0
"""
pd.read_sql_query(query,sqliteConnection)

,payment_type,total_value
0,boleto,2.869361e+06
1,credit_card,1.254208e+07


In [75]:
query = """ 
SELECT p.payment_value,
        r.review_score
FROM order_payments p
JOIN order_reviews r
ON p.order_id = r.order_id
"""
pd.read_sql_query(query,sqliteConnection)

,payment_value,review_score
0,99.33,1
1,24.39,5
2,65.71,5
3,107.78,5
4,107.78,5
...,...,...
103672,363.31,5
103673,96.80,5
103674,47.77,1
103675,369.54,5


what are the top 10 cities with highest number of customers in the dataset?

In [ ]:
query = """
SELECT customer_city, COUNT(customer_id) AS total_customer
FROM customers
GROUP BY customer_city
ORDER BY total_customer DESC
LIMIT 10
"""
pd.read_sql_query(query,sqliteConnection)

In [76]:
query=""" 
SELECT DISTINCT customer_city
FROM customers
WHERE customer_city LIKE 'sao %'
LIMIT 10
"""
pd.read_sql_query(query,sqliteConnection)

,customer_city
0,sao bernardo do campo
1,sao paulo
2,sao jose dos campos
3,sao roque
4,sao luis
5,sao jose
6,sao jose dos pinhais
7,sao vicente
8,sao goncalo
9,sao carlos


In [77]:
query = """ 
SELECT DISTINCT product_category_name as name
FROM products
WHERE name LIKE'%moveis%'
"""
pd.read_sql_query(query,sqliteConnection)

,name
0,moveis_decoracao
1,moveis_escritorio
2,moveis_sala
3,moveis_cozinha_area_de_servico_jantar_e_jardim
4,moveis_quarto
5,moveis_colchao_e_estofado


How many unique product categories does the e-commerce dataset contains?

In [88]:
query = """ 
SELECT  COUNT(DISTINCT product_category_name) as total_categories
FROM products 

"""
pd.read_sql_query(query,sqliteConnection)

##alternative
    # """ SELECT COUNT(*)
    #     FROM product_category_name_translation
    # """

,total_categories
0,73


e want to find our biggest single transactions. Show me all payments where the customer spent more than $1,000 in a single transaction.

In [91]:
query=""" 
SELECT order_id,payment_value 
FROM order_payments
WHERE payment_value > 1000.0
ORDER BY payment_value DESC
"""
pd.read_sql_query(query,sqliteConnection)

,order_id,payment_value
0,03caa2c082116e1d31e67e9ae3700499,13664.08
1,736e1922ae60d0d6a89247b851902527,7274.88
2,0812eb902a67711a1cb742b3cdaa65ae,6929.31
3,fefacc66af859508bf1a7934eab1e97f,6922.21
4,f5136e38d1a14a4dbd87dff67da82701,6726.66
...,...,...
1145,c40cbf2797c3b0187b41ea072dcac55f,1003.42
1146,491fc1fb624207b2123368e6b0ea4df1,1003.08
1147,fa794cef99ed64f42ebb766a8af51c64,1002.79
1148,62d9b911d7c56cf455f660eecb8ddd3a,1002.73


What is the average number of payment installments customers use when paying by credit card?

In [96]:
query=""" 
SELECT ROUND(AVG(payment_installments),1) as avg_installments
FROM order_payments
WHERE payment_type = 'credit_card'
"""
pd.read_sql_query(query,sqliteConnection)


,avg_installments
0,3.5


Which 5 product categories have the heaviest products on average?

In [ ]:
query=""" 
SELECT product_category_name as category, 
        AVG(product_weight_g) as sum_weight_products
FROM products
GROUP BY category
ORDER BY sum_weight_products DESC
LIMIT(5)
"""
pd.read_sql_query(query,sqliteConnection)

,category,sum_weight_products
0,moveis_decoracao,7992965.0
1,cama_mesa_banho,7440451.0
2,utilidades_domesticas,7053553.0
3,esporte_lazer,5804731.0
4,automotivo,5043836.0


 Sometimes customers split payments (e.g., a voucher and a credit card). Find all the order_ids that have more than one payment record associated with them

In [116]:
query = """ 
SELECT  order_id , COUNT(order_id) as count
FROM order_payments
GROUP BY order_id
HAVING count >= 2
ORDER BY count DESC
"""
pd.read_sql_query(query,sqliteConnection)

,order_id,count
0,fa65dad1b0e818e3ccc5cb0e39231352,29
1,ccf804e764ed5650cd8759557269dc13,26
2,285c2e15bebd4ac83635ccc563dc71f4,22
3,895ab968e7bb0d5659d16cd74cd1650c,21
4,fedcd9f7ccdc8cba3a18defedd1a5547,19
...,...,...
2956,00e6bc6b166eb28b4502c1cad4457248,2
2957,00b4a910f64f24dbcac04fe54088a443,2
2958,0071ee2429bc1efdc43aa3e073a5290e,2
2959,002f19a65a2ddd70a090297872e6d64e,2


What is the total revenue (sum of payment value) generated strictly by orders that received a perfect 5-star review?

In [6]:
query=""" 
SELECT SUM(p.payment_value) as total_5_star_revenue
FROM order_reviews r
JOIN order_payments p
ON p.order_id = r.order_id
WHERE r.review_score = 5
"""

pd.read_sql_query(query,sqliteConnection)

,total_5_star_revenue
0,8955199.96


Which 3 cities have the highest concentration of sellers?

In [123]:
query = """
SELECT seller_city, COUNT( DISTINCT seller_id) as count
FROM sellers
GROUP BY seller_city
ORDER BY count DESC
LIMIT(3)
"""
pd.read_sql_query(query,sqliteConnection)

,seller_city,count
0,sao paulo,694
1,curitiba,127
2,rio de janeiro,96


Which 5 customer cities have generated the most total revenue for the company?

In [7]:
query =""" 
SELECT c.customer_city, SUM(p.payment_value) as sum
FROM orders o
JOIN customers c
ON c.customer_id = o.customer_id
JOIN order_payments p
ON p.order_id = o.order_id
GROUP BY c.customer_city
ORDER BY sum DESC
LIMIT(5)
"""
pd.read_sql_query(query,sqliteConnection)

,customer_city,sum
0,sao paulo,2203373.09
1,rio de janeiro,1161927.36
2,belo horizonte,421765.12
3,brasilia,354216.78
4,curitiba,247392.48


he product team wants to fix bad products. Find the top 5 product categories with the lowest average review score. Crucial restriction: Only include categories that have received at least 30 total reviews to filter out outliers.

In [8]:
query=""" 
SELECT p.product_category_name, AVG(r.review_score) as avg, COUNT(r.review_score) as count
FROM order_items i
JOIN order_reviews r
ON r.order_id=i.order_id
JOIN products p
on p.product_id = i.product_id
GROUP BY p.product_category_name
HAVING count >=30
ORDER BY avg ASC
LIMIT (5)
"""

pd.read_sql_query(query,sqliteConnection)

,product_category_name,avg,count
0,fraldas_higiene,3.256410,39
1,moveis_escritorio,3.493183,1687
2,fashion_roupa_masculina,3.641221,131
3,telefonia_fixa,3.683206,262
4,artigos_de_festas,3.767442,43


 Is the average review score higher for orders delivered before their estimated delivery date compared to orders delivered after?

In [11]:
query=""" 
SELECT AVG(r.review_score),
CASE 
    WHEN julianday(o.order_estimated_delivery_date) - julianday(o.order_delivered_customer_date) >= 0 THEN 'BEFORE'
    WHEN julianday(o.order_estimated_delivery_date) - julianday(o.order_delivered_customer_date) < 0 THEN 'AFTER'
    ELSE 'AFTER'
    END AS delivery_status
FROM orders o
JOIN order_reviews r
ON o.order_id = r.order_id
GROUP BY delivery_status
"""
pd.read_sql_query(query,sqliteConnection)

,AVG(r.review_score),delivery_status
0,2.348192,AFTER
1,4.293578,BEFORE


In [142]:
sqliteConnection.close()